In [159]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

In [160]:
folder = "Data_Set"  
customer_info = pd.read_csv(os.path.join(folder, "customer_info.csv"))
customer_product = pd.read_csv(os.path.join(folder, "customer_product.csv"))
customer_cases = pd.read_csv(os.path.join(folder, "customer_cases.csv"))
product_info = pd.read_csv(os.path.join(folder, "product_info.csv"))

print("Data loaded successfully.")

Data loaded successfully.


## Basis idea about all 4 data_set

In [161]:
display(customer_info.head(4))
display(customer_info.info())
display(customer_info.isna().sum())
display(customer_info.duplicated().sum())

,Unnamed: 0,customer_id,age,gender
0,1,C2448,76,female
1,2,C2449,61,male
2,3,C2450,58,female
3,4,C2451,62,female


<class 'pandas.DataFrame'>
RangeIndex: 508932 entries, 0 to 508931
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   Unnamed: 0   508932 non-null  int64
 1   customer_id  508932 non-null  str  
 2   age          508932 non-null  int64
 3   gender       508932 non-null  str  
dtypes: int64(2), str(2)
memory usage: 15.5 MB


None

Unnamed: 0     0
customer_id    0
age            0
gender         0
dtype: int64

np.int64(0)

In [162]:
display(customer_cases.head(4))
display(customer_cases.info())
display(customer_cases.isna().sum())
display(customer_cases.duplicated().sum())

,Unnamed: 0,case_id,date_time,customer_id,channel,reason
0,1,CC101,2017-01-01 10:32:03,C2448,phone,signup
1,2,CC102,2017-01-01 11:35:47,C2449,phone,signup
2,3,CC103,2017-01-01 11:37:09,C2450,phone,signup
3,4,CC104,2017-01-01 13:28:14,C2451,phone,signup


<class 'pandas.DataFrame'>
RangeIndex: 330512 entries, 0 to 330511
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   Unnamed: 0   330512 non-null  int64
 1   case_id      330512 non-null  str  
 2   date_time    330512 non-null  str  
 3   customer_id  330512 non-null  str  
 4   channel      330512 non-null  str  
 5   reason       330512 non-null  str  
dtypes: int64(1), str(5)
memory usage: 15.1 MB


None

Unnamed: 0     0
case_id        0
date_time      0
customer_id    0
channel        0
reason         0
dtype: int64

np.int64(0)

In [163]:
display(customer_product.head(4))
display(customer_product.info())
display(customer_product.isna().sum())
display(customer_product.duplicated().sum())

,Unnamed: 0,customer_id,product,signup_date_time,cancel_date_time
0,1,C2448,prd_1,2017-01-01 10:35:09,NaN
1,2,C2449,prd_1,2017-01-01 11:39:29,2021-09-05 10:00:02
2,3,C2450,prd_1,2017-01-01 11:42:00,2019-01-13 16:24:55
3,4,C2451,prd_2,2017-01-01 13:32:08,NaN


<class 'pandas.DataFrame'>
RangeIndex: 508932 entries, 0 to 508931
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   Unnamed: 0        508932 non-null  int64
 1   customer_id       508932 non-null  str  
 2   product           508932 non-null  str  
 3   signup_date_time  508932 non-null  str  
 4   cancel_date_time  112485 non-null  str  
dtypes: int64(1), str(4)
memory usage: 19.4 MB


None

Unnamed: 0               0
customer_id              0
product                  0
signup_date_time         0
cancel_date_time    396447
dtype: int64

np.int64(0)

In [164]:
display(product_info.head(4))
display(product_info.info())
display(product_info.isna().sum())
display(product_info.duplicated().sum())

,product_id,name,price,billing_cycle
0,prd_1,annual_subscription,1200,12
1,prd_2,monthly_subscription,125,1


<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   product_id     2 non-null      str  
 1   name           2 non-null      str  
 2   price          2 non-null      int64
 3   billing_cycle  2 non-null      int64
dtypes: int64(2), str(2)
memory usage: 196.0 bytes


None

product_id       0
name             0
price            0
billing_cycle    0
dtype: int64

np.int64(0)

1. As we can see in the [customer_cases, customer_info, and product_info] datasets, there is an unnecessary column named **`Unnamed: 0`**, which needs to be removed.

2. In the **customer_info** dataset, the **date-time column** has a string (`str`) data type, so we need to convert it to **datetime** for future analysis. Similarly, in the **customer_product** dataset, the **`signup_date`** and **`cancel_date`** columns are stored as strings instead of datetime, so they also need to be converted to the **datetime** data type.


# Data Wranling Process

In [165]:
# 1. Drop unnecessary index columns
for df in [customer_info, customer_product, customer_cases]:
    if 'Unnamed: 0' in df.columns:
        df.drop(columns=['Unnamed: 0'], inplace=True)

# 2. Correct Data Types
customer_cases['date_time'] = pd.to_datetime(customer_cases['date_time'])
customer_product['signup_date_time'] = pd.to_datetime(customer_product['signup_date_time'])
customer_product['cancel_date_time'] = pd.to_datetime(customer_product['cancel_date_time'])

# 3. Create Churn Flag
customer_product['is_cancelled'] = customer_product['cancel_date_time'].notna()

# 4. FIX: Calculate Subscription Duration using Maximum Dataset Date (Prevents Data Leakage)
max_dataset_date = customer_product['signup_date_time'].max()
end_date = customer_product['cancel_date_time'].fillna(max_dataset_date)
customer_product['subscription_duration_days'] = (end_date - customer_product['signup_date_time']).dt.days

print("Wrangling complete. Fixed subscription duration calculations.")

Wrangling complete. Fixed subscription duration calculations.


In [166]:
cases_per_customer = customer_cases.groupby('customer_id').size().reset_index(name='total_support_cases')


df_master = customer_product.merge(customer_info, on='customer_id', how='left')
df_master = df_master.merge(product_info, left_on='product', right_on='product_id', how='left')
df_master = df_master.merge(cases_per_customer, on='customer_id', how='left')

df_master['total_support_cases'] = df_master['total_support_cases'].fillna(0).astype(int)


df_master['months_active'] = np.maximum(1, np.ceil(df_master['subscription_duration_days'] / 30))
df_master['estimated_revenue'] = np.where(
    df_master['product'] == 'prd_1',
    df_master['price'] * np.maximum(1, np.ceil(df_master['months_active'] / 12)),
    df_master['price'] * df_master['months_active']
)

print(f"Master Dataset Created. Shape: {df_master.shape}")
df_master.head()

Master Dataset Created. Shape: (508932, 15)


,customer_id,product,signup_date_time,cancel_date_time,is_cancelled,subscription_duration_days,age,gender,product_id,name,price,billing_cycle,total_support_cases,months_active,estimated_revenue
0,C2448,prd_1,2017-01-01 10:35:09,NaT,False,1825,76,female,prd_1,annual_subscription,1200,12,1,61.0,7200.0
1,C2449,prd_1,2017-01-01 11:39:29,2021-09-05 10:00:02,True,1707,61,male,prd_1,annual_subscription,1200,12,1,57.0,6000.0
2,C2450,prd_1,2017-01-01 11:42:00,2019-01-13 16:24:55,True,742,58,female,prd_1,annual_subscription,1200,12,1,25.0,3600.0
3,C2451,prd_2,2017-01-01 13:32:08,NaT,False,1825,62,female,prd_2,monthly_subscription,125,1,2,61.0,7625.0
4,C2452,prd_1,2017-01-01 13:57:30,2021-06-28 18:06:01,True,1639,71,male,prd_1,annual_subscription,1200,12,1,55.0,6000.0


In [167]:
df_master.head()

,customer_id,product,signup_date_time,cancel_date_time,is_cancelled,subscription_duration_days,age,gender,product_id,name,price,billing_cycle,total_support_cases,months_active,estimated_revenue
0,C2448,prd_1,2017-01-01 10:35:09,NaT,False,1825,76,female,prd_1,annual_subscription,1200,12,1,61.0,7200.0
1,C2449,prd_1,2017-01-01 11:39:29,2021-09-05 10:00:02,True,1707,61,male,prd_1,annual_subscription,1200,12,1,57.0,6000.0
2,C2450,prd_1,2017-01-01 11:42:00,2019-01-13 16:24:55,True,742,58,female,prd_1,annual_subscription,1200,12,1,25.0,3600.0
3,C2451,prd_2,2017-01-01 13:32:08,NaT,False,1825,62,female,prd_2,monthly_subscription,125,1,2,61.0,7625.0
4,C2452,prd_1,2017-01-01 13:57:30,2021-06-28 18:06:01,True,1639,71,male,prd_1,annual_subscription,1200,12,1,55.0,6000.0


In [168]:
# Export the clean master dataset to CSV
df_master.to_csv("Processed_Data_Master.csv", index=False)
print("Saved 'Processed_Data_Master.csv' for import into Google Sheets.")

Saved 'Processed_Data_Master.csv' for import into Google Sheets.


>What is our current active subscriber count, and what is the Annual Recurring Revenue (ARR) contribution split between annual subscriptions (prd_1) and monthly subscriptions (prd_2)?

In [169]:

active_subscribers = df_master[df_master['is_cancelled'] == False]
q1_arr = active_subscribers.groupby('product').agg(
    active_users=('customer_id', 'count')
).reset_index()
arr_mapping = {'prd_1': 1200, 'prd_2': 1500}
q1_arr['arr_per_user'] = q1_arr['product'].map(arr_mapping)
q1_arr['total_arr'] = q1_arr['active_users'] * q1_arr['arr_per_user']

total_platform_arr = q1_arr['total_arr'].sum()
q1_arr['arr_share_%'] = (q1_arr['total_arr'] / total_platform_arr) * 100

q1_arr = q1_arr.drop(columns=['arr_per_user'])
display(q1_arr)
print(f"Total Platform ARR: ${total_platform_arr:,.2f}")

,product,active_users,total_arr,arr_share_%
0,prd_1,255967,307160400,59.311069
1,prd_2,140480,210720000,40.688931


Total Platform ARR: $517,880,400.00


>How significantly does the churn rate of monthly subscribers (prd_2) differ from annual subscribers (prd_1), and what is the resulting loss in Customer Lifetime Value (LTV)?

In [170]:
churn_df = df_master.groupby('product_id').agg(
    total_users=('is_cancelled', 'count'),
    churned_users=('is_cancelled', 'sum'),
    avg_days=('subscription_duration_days', 'mean')
).reset_index()

churn_df['active_users'] = churn_df['total_users'] - churn_df['churned_users']
churn_df['churn_rate'] = (churn_df['churned_users'] / churn_df['total_users']) * 100
churn_df['avg_months'] = churn_df['avg_days'] / 30

monthly_rates = {'prd_1': 100, 'prd_2': 125}
churn_df['monthly_rate'] = churn_df['product_id'].map(monthly_rates)
churn_df['avg_ltv'] = churn_df['monthly_rate'] * churn_df['avg_months']

churn_df = churn_df.drop(columns=['avg_days', 'monthly_rate'])
churn_df

,product_id,total_users,churned_users,active_users,churn_rate,avg_months,avg_ltv
0,prd_1,325649,69682,255967,21.397885,22.04086,2204.086005
1,prd_2,183283,42803,140480,23.353503,14.78594,1848.242540


>Do subscribers who contact support multiple times exhibit a higher probability of cancelling compared to those who rarely or never log support cases?

In [171]:
cases_summary = df_master.groupby('total_support_cases').agg(
    total_users=('is_cancelled', 'count'),
    churned_users=('is_cancelled', 'sum')
).reset_index()

# Derive active count and churn percentage
cases_summary['active_users'] = cases_summary['total_users'] - cases_summary['churned_users']
cases_summary['churn_rate'] = (cases_summary['churned_users'] / cases_summary['total_users']) * 100

cases_summary = cases_summary[['total_support_cases', 'active_users', 'churned_users', 'total_users', 'churn_rate']]
cases_summary

,total_support_cases,active_users,churned_users,total_users,churn_rate
0,0,197365,52907,250272,21.139800
1,1,150050,44407,194457,22.836411
2,2,43545,13335,56880,23.444093
3,3,5264,1737,7001,24.810741
4,4,221,97,318,30.503145
5,5,2,2,4,50.000000


>What are the top reasons customers reach out to support (e.g., signup, cancellation, technical support), and through which primary channels (e.g., phone, email, chat) do these cases originate?

In [172]:
# 1. Total count and percentage by support reason
reason_summary = customer_cases['reason'].value_counts().reset_index()
reason_summary.columns = ['reason', 'total_cases']
reason_summary['percent'] = (reason_summary['total_cases'] / len(customer_cases)) * 100

# 2. Total count and percentage by support channel
channel_summary = customer_cases['channel'].value_counts().reset_index()
channel_summary.columns = ['channel', 'total_cases']
channel_summary['percent'] = (channel_summary['total_cases'] / len(customer_cases)) * 100

# 3. Breakdown matrix (Reason vs Channel)
reason_by_channel = pd.crosstab(customer_cases['reason'], customer_cases['channel'],normalize='index')*100

display(reason_summary)
display(channel_summary)
display(reason_by_channel)

,reason,total_cases,percent
0,support,200985,60.810198
1,signup,129527,39.189802


,channel,total_cases,percent
0,phone,286840,86.786561
1,email,43672,13.213439


channel,email,phone
reason,,
signup,0.000000,100.000000
support,21.728985,78.271015


>How do cancellation rates vary across different age brackets (e.g., 18–30, 31–45, 46–60, 60+), and which age group represents our highest churn risk?

In [173]:
bins = [17, 30, 45, 60, 100]
labels = ['18-30', '31-45', '46-60', '60+']
df_master['age_group'] = pd.cut(df_master['age'], bins=bins, labels=labels)

# 2. Group by age group to calculate total, active, churned, and cancellation rate
age_summary = df_master.groupby('age_group').agg(
    total_users=('is_cancelled', 'count'),
    churned_users=('is_cancelled', 'sum')
).reset_index()

age_summary['active_users'] = age_summary['total_users'] - age_summary['churned_users']
age_summary['cancellation_rate_%'] = (age_summary['churned_users'] / age_summary['total_users']) * 100

age_summary

,age_group,total_users,churned_users,active_users,cancellation_rate_%
0,18-30,1307,296,1011,22.647284
1,31-45,32810,7587,25223,23.124048
2,46-60,256093,56421,199672,22.031450
3,60+,218722,48181,170541,22.028420


### Hypotheses Test

>Hypothesis 1: Monthly subscribers (prd_2) have a significantly higher churn rate (>35%) than annual subscribers (prd_1) because the lower commitment fee results in lower long-term platform engagement.

In [177]:
# Hypothesis 1: Monthly subscribers churn rate > 35%
hyp1_results = df_master.groupby('product').agg(
    total_users=('is_cancelled', 'count'),
    churned_users=('is_cancelled', 'sum')
).reset_index()

hyp1_results['churn_rate_%'] = (hyp1_results['churned_users'] / hyp1_results['total_users']) * 100
display(hyp1_results)

# Verdict printout
prd2_rate = hyp1_results.loc[hyp1_results['product'] == 'prd_2', 'churn_rate_%'].values[0]
print(f"Monthly Plan Churn: {prd2_rate:.2f}% | Hypothesis (>35%): Disproven")

,product,total_users,churned_users,churn_rate_%
0,prd_1,325649,69682,21.397885
1,prd_2,183283,42803,23.353503


Monthly Plan Churn: 23.35% | Hypothesis (>35%): Disproven


>Hypothesis 2: Subscribers who log 3 or more support tickets within their customer lifetime exhibit at least a 40% higher cancellation rate than those with fewer than 3 tickets, indicating product usage friction.

In [176]:
# Hypothesis 2: 3+ support tickets leads to >= 40% higher churn rate
df_master['high_support_volume'] = df_master['total_support_cases'] >= 3

hyp2_results = df_master.groupby('high_support_volume').agg(
    total_users=('is_cancelled', 'count'),
    churned_users=('is_cancelled', 'sum')
).reset_index()

hyp2_results['churn_rate_%'] = (hyp2_results['churned_users'] / hyp2_results['total_users']) * 100
hyp2_results['group_label'] = hyp2_results['high_support_volume'].map({False: '< 3 Cases', True: '3+ Cases'})

display(hyp2_results[['group_label', 'total_users', 'churned_users', 'churn_rate_%']])

rate_low = hyp2_results.loc[hyp2_results['group_label'] == '< 3 Cases', 'churn_rate_%'].values[0]
rate_high = hyp2_results.loc[hyp2_results['group_label'] == '3+ Cases', 'churn_rate_%'].values[0]
relative_increase = ((rate_high - rate_low) / rate_low) * 100

print(f"Relative Increase in Churn: {relative_increase:.2f}% | Hypothesis (>=40%): Disproven")

,group_label,total_users,churned_users,churn_rate_%
0,< 3 Cases,501609,110649,22.058815
1,3+ Cases,7323,1836,25.071692


Relative Increase in Churn: 13.66% | Hypothesis (>=40%): Disproven


In [175]:
# Sample 50,000 rows from your master Pandas dataframe
df_sample = df_master.sample(n=50000, random_state=42)
df_sample.to_csv("master_data_sample.csv", index=False)